# Двухбашенный кандген: обучение на GPU (DataSphere)

Что это и как устроено — `docs/twotower.md`. Ноутбук только запускает скрипты из корня репозитория:
`scripts/train_twotower.py` (обучение) и `scripts/encode_twotower.py` (векторы треков и запросов).
Параметры — `configs/twotower.yaml`, любой можно переопределить через `--set`.

Нужно в корне репозитория: `music4all_crs/` (train, test_public, tracks_meta) и `artifacts/semantic_ids/`
(`semantic_codes.npy`, `rqvae_checkpoint.pt`).

In [ ]:
# пакеты для обучения (torch на GPU-конфигурации DataSphere обычно уже стоит)
%pip install -q "transformers>=4.51" "peft>=0.13"

In [ ]:
import os, torch
assert os.path.exists("scripts/train_twotower.py"), "откройте ноутбук из корня репозитория"
for p in ["music4all_crs/train.parquet", "music4all_crs/tracks_meta.parquet",
          "artifacts/semantic_ids/semantic_codes.npy", "artifacts/semantic_ids/rqvae_checkpoint.pt"]:
    print(("ok  " if os.path.exists(p) else "НЕТ ") + p)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "нет")
if torch.cuda.is_available() and not torch.cuda.is_bf16_supported():
    print("bf16 нет (V100): добавьте --set twotower.model.precision=fp16")

## 1. Быстрая проверка (~минуты)
300 пользователей, по 20 шагов на этап: всё скачивается, грузится и считается без ошибок; заодно видно скорость шага.

In [ ]:
!python scripts/train_twotower.py --run-dir artifacts/twotower/smoke --n-users 300 --set twotower.train.max_steps=20

## 2. Основная модель
Этап A (Qwen заморожен) и этап B (+ LoRA на attention). Лог — `artifacts/twotower/main/log.jsonl`.
Если не влезает в память — `--set twotower.train.batch_size=64 --set twotower.train.grad_accum=2`.

In [ ]:
!python scripts/train_twotower.py --run-dir artifacts/twotower/main

In [ ]:
import json
m = json.load(open("artifacts/twotower/main/metrics.json"))
print("лучшее на валидации:", m["best_val"])
{t: round(v["recall@100"], 3) for t, v in m["last_val"]["by_type"].items()}

## 3. Векторы для пайплайна
Треки и все запросы test_public -> `artifacts/twotower/main/`. Дальше GPU не нужен: папку можно скачать и считать
метрики локально (`scripts/evaluate_candgen.py --set retrieval.twotower.enabled=true`).

In [ ]:
!python scripts/encode_twotower.py --run-dir artifacts/twotower/main
for L in (0, 16):   # ablation длины истории без переобучения
    !python scripts/encode_twotower.py --run-dir artifacts/twotower/main --history-len {L}

In [ ]:
!python scripts/evaluate_candgen.py --n-users 1000 --offline --set retrieval.twotower.enabled=true --set retrieval.twotower.dir=artifacts/twotower/main

## 4. Ablation
Каждый — отдельная папка; дальше так же `encode_twotower.py` и `evaluate_candgen.py`. Порядок — по пользе.

In [ ]:
# без LoRA: только этап A, столько же шагов
!python scripts/train_twotower.py --run-dir artifacts/twotower/no_lora --set 'twotower.train.stages=[{name: A, epochs: 1.3, lora: false, lr: 1.0e-3}]'

In [ ]:
# случайные коды вместо RQ-VAE (semantic ID против хеш-ID)
!python scripts/train_twotower.py --run-dir artifacts/twotower/random_codes --set twotower.codes.source=random --set twotower.codes.init=random

In [ ]:
# in-batch негативы без и с поправкой logQ
!python scripts/train_twotower.py --run-dir artifacts/twotower/in_batch --set twotower.loss.negatives=in_batch
!python scripts/train_twotower.py --run-dir artifacts/twotower/in_batch_logq --set twotower.loss.negatives=in_batch --set twotower.loss.logq=true